In [2]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd
from corpus.swc_mono_test import SWCMonoTestSetH5Dataset

from corpus.speaker_room_dataset import SpeakerRoomDataset
import src.audio_transforms as at
import scipy.stats as stats

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer
from tqdm.auto import tqdm
import src.spatial_attn_architecture as attn_arch
import re
import src.custom_modules as cm

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# torch.set_float32_matmul_precision('medium')
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

In [ ]:
!hostname

node109


In [3]:
### Get most recent config
# config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
# ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=3-step=49432.ckpt"
# old_style = False
### Get most recent config
model_name = "word_task_half_co_loc_v09_gender_bal_4M_w_no_cue_learned_higher_lr_less_dropout"
config_path = f"config/binaural_attn/{model_name}.yaml"
ckpt_path = f"attn_cue_models/{model_name}/checkpoints/epoch=2-step=35108-v1.ckpt"

old_style = False 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['hparas']['batch_size'] = 2 # config['data']['loader']['batch_size'] // args.gpus
config['num_workers'] = 2
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
cue_duration = 2
n_cue_frames = int(cue_duration * signal_sr)
config['model']['n_cue_frames'] = n_cue_frames

config['cue_duration_test'] = True


In [ ]:
model_name

# get version str from model name 
import re 

int(re.search("v\d+", model_name).group(0).strip('v'))

9

In [5]:
# int(torch.__version__.split('.')[0])
model = binaural_lightning.BinauralAttentionModule(config)

Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using cue duration of 88200 frames
Using cue duration of 22042 frames
Using cue duration of 5508 frames
Using cue duration of 1101 frames
Using cue duration of 275 frames
Using cue duration of 270 frames
Using cue duration of 266 frames
Using cue duration of 66 frames
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [ ]:
# model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config,)
model = model.eval()

Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using cue duration of 88200 frames
Using cue duration of 22042 frames
Using cue duration of 5508 frames
Using cue duration of 1101 frames
Using cue duration of 275 frames
Using cue duration of 270 frames
Using cue duration of 266 frames
Using cue duration of 66 frames
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


RuntimeError: Error(s) in loading state_dict for BinauralAttentionModule:
	Missing key(s) in state_dict: "model.model_dict.norm_coch_rep.weight", "model.model_dict.norm_coch_rep.bias", "model.model_dict.attn0.bias", "model.model_dict.attn0.slope", "model.model_dict.attn0.threshold", "model.model_dict.conv_block_0.0.weight", "model.model_dict.conv_block_0.0.bias", "model.model_dict.conv_block_0.1.weight", "model.model_dict.hann_pool_0.hann_window2d", "model.model_dict.attn1.bias", "model.model_dict.attn1.slope", "model.model_dict.attn1.threshold", "model.model_dict.conv_block_1.0.weight", "model.model_dict.conv_block_1.0.bias", "model.model_dict.conv_block_1.1.weight", "model.model_dict.hann_pool_1.hann_window2d", "model.model_dict.attn2.bias", "model.model_dict.attn2.slope", "model.model_dict.attn2.threshold", "model.model_dict.conv_block_2.0.weight", "model.model_dict.conv_block_2.0.bias", "model.model_dict.conv_block_2.1.weight", "model.model_dict.hann_pool_2.hann_window2d", "model.m

In [ ]:
model

In [53]:
# # get layer norm parameters from model 
# model.model._orig_mod.model_dict.norm_coch_rep.weight.shape
# isinstance(model.model._orig_mod.model_dict.norm_coch_rep, torch.nn.LayerNorm)

# new_weight = model.model._orig_mod.model_dict.norm_coch_rep.state_dict()
# new_weight

In [54]:
# from src.spatial_attn_architecture import CueDurationCNN2DExtractor, CueDurationCNNNew 

# cue_dur_in_s = 0.2
# cue_dur = int(cue_dur_in_s * coch_sr) 


# if old_style:
#     model = CueDurationCNN2DExtractor(**config['model'], n_cue_frames=cue_dur)
# else:
#     model = CueDurationCNNNew(**config['model'], n_cue_frames=cue_dur)

# # restore params from checkpoint 
# checkpoint = torch.load(ckpt_path)
# checkpoint['state_dict'] = {k.replace('model._orig_mod.', ''):v for k,v in checkpoint['state_dict'].items()}

# # below is naming update to map checkpoint parameters to new model dict names
# new_state_dict = {}
# # old_style = True 
# for k,v in  checkpoint['state_dict'].items():
#     # print(k)
#     if config['model'].get('ln_affine', True):
#         # update key for easy norm layer access
#         if 'conv' in k and '.0.' in k:
#             k = k.replace('conv_block_', 'layer_norm_')
#             k = k.replace('.0.', '.')
#             print(k)
#         # decrement conv layer ixs in dict to match model
#         elif 'conv' in k and '.1.' in k:
#             k = k.replace('.1.', '.0.')
#         # decrement pool layer rixs in dict to match model
#         elif 'conv' in k and '.3.' in k:
#             k = k.replace('.3.', '.2.')
#     # if old_style:
#     #     if 'attn' in k:
#     #         if 'in' in k:
#     #             k = k.replace('_block_in', '0')
#     #         # if digit in string, add 1 to that digit 
#     #         else:
#     #             print(k)
#     #             layer_num = int(re.search('attn(-?\d+)', k).group(0).strip('attn'))
#     #             k = k.replace(str(layer_num), str(layer_num + 1 ))

#     new_state_dict[k] = v

# norm_param_dict = {k:v for k,v in new_state_dict.items() if 'norm' in k}

In [55]:
# norm_param_dict.keys()

In [56]:
# for key, param in norm_param_dict.items():
#     layer_name = key.split('.')[1]
#     n_cue_frames_at_layer = model.layer_norm_params[f'{layer_name}']['ln_cue_frames']
#     param_type = 'weight' if 'weight' in key else 'bias'
#     model.layer_norm_params[f'{layer_name}'][f'{param_type}'] = param
#     model.layer_norm_params[f'{layer_name}'][f'cue_{param_type}'] = param[..., : n_cue_frames_at_layer]
        

In [57]:
#  model.layer_norm_params

In [58]:
# model.load_state_dict(new_state_dict, strict=False) # strict=False to skip attn weights 



In [59]:
## Want to restore parameters from original model to new model
## For all layers, restore weights and biases from model_dict.conv_block_{idx}.norm to model_dict.mixture_norm_{idx} and model_dict.cue_norm_{idx}

# for k,v in model.model_dict.items():
#     print(k)
#     if 'norm' in k:



In [60]:
# dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
#                             excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
#                             cue_type='voice_and_location',
#                             sr=signal_sr) 

dataset = SWCMonoTestSetH5Dataset(h5_path="/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5",
                                            eval_distractor_cond="1-talker-english-different",
                                            model_sr=44100,
                                            label_type='CV')
    

diotic_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), 
                    at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                    at.DuplicateChannel(),

            ])


diotic_transforms = diotic_transforms.cuda()


def single_signal_collate_fn(batch):
    #apply transforsms to batch
    cues = torch.stack([diotic_transforms(cue, None)[0] for cue, fg, bg, label, confusion in batch])
    mixtures = torch.stack([diotic_transforms(fg, bg)[0] for cue, fg, bg, label, confusion in batch]).type(torch.FloatTensor)
    labels = torch.tensor([label for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    confusion = torch.tensor([confusion for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    return cues, mixtures, labels, confusion

dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False, num_workers=config['num_workers'], collate_fn=single_signal_collate_fn)


In [61]:
# torch module to center crop the audio to a given duration 
class CenterCropCue(torch.nn.Module):
    def __init__(self, crop_duration, sig_duration, sr, pad_dur=False):
        super().__init__()
        self.n_crop_frames = int(crop_duration * sr)
        sig_frames = int(sig_duration * sr)
        self.crop_context = sig_frames - self.n_crop_frames
        self.crop_start = self.crop_context // 2
        self.crop_end = self.crop_start + self.n_crop_frames
        self.pad_to_dur = False
        
        if pad_dur > crop_duration:
            self.pad_to_dur = True 
            self.pad_context = int(pad_dur * sr) - self.n_crop_frames
        print(f"crop_start: {self.crop_start}, crop_end: {self.crop_end}, pad_context: {self.pad_context}")

    def forward(self, x):
        # crop x to n_crop_frames and zero pad back to original length
        cropped = x[..., self.crop_start:self.crop_end]
        if self.pad_to_dur:
            cropped = torch.nn.functional.pad(cropped, (0,self.pad_context), "constant", 0)
        return cropped 
    
class StartCropCue(torch.nn.Module):
    def __init__(self, crop_duration, sig_duration, sr, pad_dur=False):
        super().__init__()
        self.n_crop_frames = int(crop_duration * sr)
        sig_frames = int(sig_duration * sr)
        if pad_dur:
            pad_frames = int(pad_dur * sr)
            self.pad_context = pad_frames - self.n_crop_frames
        else:
            self.pad_context = sig_frames - self.n_crop_frames
        print(f"crop_duration: {crop_duration}, pad_context: {self.pad_context}")

    def forward(self, x):
        # crop x to n_crop_frames and zero pad back to original length
        cropped = x[..., :self.n_crop_frames]
        cropped = torch.nn.functional.pad(cropped, (0,self.pad_context), "constant", 0)
        return cropped 



In [62]:
!hostname

node109


In [63]:
%matplotlib inline
import matplotlib.pyplot as plt

# plt.imshow(cue[0][0].cpu().numpy(), aspect='auto', origin='upper')

In [64]:
# torch.load(ckpt_path)['state_dict'].keys()  

In [65]:
coch_sr

10000

In [66]:
output_dict = {'results': None, 'confusions': None}
accuracies = []
pred_list = []
true_word_int = []
confusions_list  = []
all_probs_of_interest = []
guessed_both = []

crop_duration = .2 # cue_duration # # in seconds 
sig_duration = 2 # in seconds 
# center_crop = CenterCropCue(crop_duration, sig_duration, signal_sr, pad_dur=2).cuda() 
center_crop = StartCropCue(crop_duration, sig_duration, signal_sr, pad_dur=2).cuda() 

signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
n_cue_frames = int(crop_duration * coch_sr)
print(n_cue_frames)
config['model']['n_cue_frames'] = n_cue_frames
#
# module = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False)
coch_gram = cm.AttnAudioInputRepresentation(**config['audio']).cuda()

# model = torch.compile(model)

model = model.eval().cuda()

with torch.no_grad(): 
    for ix, batch in enumerate(tqdm(dataloader)):
        if ix > 20:
            break
        cue, mixture, label, confusion = batch
        cue = center_crop(cue)
        cue, mixture = coch_gram(cue.cuda(), mixture.cuda())
        # cue = center_crop(cue)
        # cue_mask_ixs = torch.arange(cue.shape[0])
        logits = model(cue, mixture, None)
        probs = logits.softmax(dim=-1).cpu().detach().numpy()

        # get top 2 probs for each example
        top_2_probs = torch.topk(logits.softmax(-1), 5, dim=-1).indices.cpu().detach().numpy()
        return_both = (np.isin(label.numpy(), top_2_probs) * np.isin(confusion.numpy(), top_2_probs)).astype('int')
        
        targ_probs = probs[torch.arange(probs.shape[0]), label]
        conf_probs = probs[torch.arange(probs.shape[0]), confusion]
        probs_of_interest = np.concatenate([targ_probs[:, None], conf_probs[:, None]], axis=1)

        preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
        true_word = label.numpy().astype('int')
        accuracy = (preds == true_word).astype('int')
        conf = (preds == confusion.numpy().astype('int')).astype('int')
        confusions_list.append(conf)
        accuracies.append(accuracy)
        pred_list.append(preds)
        true_word_int.append(true_word)
        all_probs_of_interest.append(probs_of_interest)
        guessed_both.append(return_both)
        
accuracies = np.concatenate(accuracies)
pred_list = np.concatenate(pred_list)
true_word_int = np.concatenate(true_word_int)
confusions_list = np.concatenate(confusions_list)
all_probs_of_interest = np.concatenate(all_probs_of_interest, axis=0)
guessed_both = np.concatenate(guessed_both)

output_dict['probs_of_interest'] = all_probs_of_interest
output_dict['results'] = accuracies
output_dict['preds'] = pred_list
output_dict['true_word_int'] = true_word_int

print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

crop_duration: 0.2, pad_context: 79380
2000
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


  0%|          | 0/61 [00:00<?, ?it/s]

 34%|███▍      | 21/61 [00:21<00:41,  1.03s/it]

Accuracy using 0.2s cue: 0.28, (0.02)
Confusions using 0.2s cue: 0.29, (0.02)
Guessed both talker words: 0.38, (0.03)


Using cue duration of 2 seconds    
Accuracy using fg as cue: 0.56, (0.01)    
Confusions using fg as cue: 0.09, (0.00)    
Guessed both talker words: 0.12, (0.01)    

Using cue duration of 0.1 seconds    
Accuracy using fg as cue: 0.50, (0.01)    
Confusions using fg as cue: 0.09, (0.01)    
Guessed both talker words: 0.11, (0.01)

In [ ]:
# print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

In [ ]:
# print(f"Using cue duration of {cue_duration} seconds")
# print(f"Accuracy using fg as cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using fg as cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")